# Lesson 27 - Instructor solutions

Attempt the learner notebook first. These are one set of possible solutions, with reasoning and
executable checks. Restart and run this notebook from the top: the worked examples below supply
the same definitions as the learner lesson. No learner answers or state are required.


## 1. An AI feature is a workflow, not just a prompt

A document assistant accepts a question, retrieves relevant text, asks a model to answer
from that evidence, validates the output, and reports sources or abstains. RAG means
retrieval-augmented generation: evidence is supplied at request time rather than learned by retraining.
Below are fictional policies. Their IDs will become citations. Keep application rules separate
from document text: retrieved text is untrusted evidence, never authority to change system instructions.
This lesson uses a deterministic client double; it demonstrates orchestration, not language intelligence.


In [ ]:
from dataclasses import dataclass, asdict
import json
import math
import re
import os

documents = [
    {"id":"leave", "text":"Leave requests require manager approval. Submit leave requests in the portal."},
    {"id":"energy", "text":"Energy readings are recorded daily. Review missing energy readings before reporting."},
    {"id":"tender", "text":"Tender documents require a signed checklist. Archive tender documents after review."},
]
assert len({d["id"] for d in documents}) == 3
print([d["id"] for d in documents])


## 2. Chunk text and preserve provenance

Models have a finite context budget; retrieval works on small pieces called chunks.
We split on whitespace into fixed word windows with overlap so neighboring chunks share context.
Real systems use token-aware/sentence-aware chunking, metadata, and document versions.
Word counts here are not model token counts. Preserve source and chunk IDs; inspect boundaries
before tuning the window. Predict how many windows eight words make with size 4, overlap 1.


In [ ]:
def chunk_words(text, size=12, overlap=3):
    if type(size) is not int or type(overlap) is not int or size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    words = text.split()
    chunks = []
    for start in range(0,len(words),size-overlap):
        chunks.append(" ".join(words[start:start+size]))
        if start + size >= len(words):
            break
    return chunks

assert chunk_words("a b c d e f g h",size=4,overlap=1) == ["a b c d","d e f g","g h"]
chunks = [{"id":f"{doc['id']}:{i}","source":doc["id"],"text":text}
          for doc in documents for i,text in enumerate(chunk_words(doc["text"]))]
print([(c["id"], len(c["text"].split())) for c in chunks])
assert chunk_words("") == []


## 3. Embeddings turn text into comparable vectors

An embedding represents text as numbers in a common space. Learned embeddings can capture
semantic relationships; they require a provider or local model. For an offline teaching substitute,
we use normalized word-count vectors over a fixed vocabulary. This is lexical retrieval, not a
learned semantic embedding: synonyms outside the vocabulary will be missed.
Cosine similarity is the dot product of unit vectors; zero vectors have similarity 0 by policy.
The same tokenizer/vocabulary must encode both queries and documents.


In [ ]:
def tokens(text):
    return re.findall(r"[a-z]+",text.casefold())

vocabulary = sorted({word for chunk in chunks for word in tokens(chunk["text"])})

def embed(text, vocabulary):
    words = tokens(text)
    counts = [float(words.count(word)) for word in vocabulary]
    norm = math.sqrt(sum(value*value for value in counts))
    return [value/norm for value in counts] if norm else counts

def similarity(left, right):
    return sum(a*b for a,b in zip(left,right,strict=True))

index = [(chunk,embed(chunk["text"],vocabulary)) for chunk in chunks]
query_vector = embed("leave approval",vocabulary)
print(len(query_vector), round(similarity(query_vector,query_vector),6))
assert math.isclose(similarity(query_vector,query_vector),1)
assert similarity(embed("unseenword",vocabulary),query_vector) == 0


## 4. Retrieve a bounded, ordered evidence set

Rank chunks by similarity and keep at most k positive-scoring results. Use chunk ID to
break ties deterministically. A positive lexical score only means shared words, not proof of
answerability. Production retrieval needs relevance evaluation, access controls before retrieval,
index versioning, and appropriate thresholds. Do not send an entire private corpus to a model.
Change the question to tender documents and predict the first source.


In [ ]:
def retrieve(question, k=2):
    if type(k) is not int or k <= 0:
        raise ValueError("k must be positive")
    query = embed(question,vocabulary)
    scored = [(similarity(query,vector),chunk) for chunk,vector in index]
    scored.sort(key=lambda item:(-item[0],item[1]["id"]))
    return [{**chunk,"score":score} for score,chunk in scored[:k] if score > 0]

hits = retrieve("leave approval",k=1)
print([(h["id"],round(h["score"],3)) for h in hits])
assert hits[0]["source"] == "leave"
assert retrieve("astronaut galaxies") == []


## 5. Structured JSON needs semantic checks

JSON parsing checks syntax, not business meaning. Reuse Lesson 26's contract idea with
standard-library validation so this lesson remains dependency-free. An answer must have exactly
three fields: nonblank answer text, a list of distinct known chunk IDs, and a boolean abstained.
An abstention must have no citations; a claimed answer needs at least one.
This verifies the output shape and citation IDs, not whether the text is actually supported.
Pydantic v2 can replace this boundary when stage 26 packages are installed.


In [ ]:
@dataclass(frozen=True)
class Answer:
    answer: str
    citations: tuple[str, ...]
    abstained: bool

def parse_answer(text, allowed_ids):
    data = json.loads(text)  # May raise JSONDecodeError.
    if not isinstance(data,dict) or set(data) != {"answer","citations","abstained"}:
        raise ValueError("Unexpected answer fields")
    if not isinstance(data["answer"],str) or not data["answer"].strip():
        raise ValueError("Answer must be nonblank text")
    citations = data["citations"]
    if not isinstance(citations,list) or not all(isinstance(c,str) for c in citations):
        raise ValueError("Citations must be strings")
    if len(set(citations)) != len(citations) or not set(citations) <= set(allowed_ids):
        raise ValueError("Duplicate or unknown citation")
    if type(data["abstained"]) is not bool:
        raise ValueError("abstained must be boolean")
    if (data["abstained"] and citations) or (not data["abstained"] and not citations):
        raise ValueError("Citations and abstention disagree")
    return Answer(data["answer"].strip(),tuple(citations),data["abstained"])

sample = json.dumps({"answer":"Manager approval is required.","citations":["leave:0"],"abstained":False})
assert parse_answer(sample,{"leave:0"}).citations == ("leave:0",)


## 6. A configurable client boundary

A client object hides provider-specific transport and returns raw JSON text. Configuration
and injected clients let tests use a fake without keys. This mock returns the top evidence text;
it cannot reason or defend against prompt injection. Real adapters must handle authentication,
timeouts, bounded retries for transient errors, rate limits, response envelopes, and cost budgets.
Do not retry malformed application output indefinitely. Never log API keys or full sensitive prompts.


In [ ]:
@dataclass(frozen=True)
class ClientConfig:
    model: str = "offline-mock"
    timeout_seconds: float = 10.
    max_output_chars: int = 2000

class MockClient:
    def __init__(self, config=None):
        self.config = config or ClientConfig()
    def complete(self, *, question, evidence):
        if not evidence:
            return json.dumps({"answer":"I do not have supporting evidence.","citations":[],"abstained":True})
        first = evidence[0]
        return json.dumps({"answer":first["text"],"citations":[first["id"]],"abstained":False})

config = ClientConfig(model=os.environ.get("COURSE_LLM_MODEL","offline-mock"))
mock = MockClient(config)
print(config.model)
assert parse_answer(mock.complete(question="leave",evidence=hits),{h["id"] for h in hits}).abstained is False


## 7. Compose retrieval, generation, validation, and failure handling

The workflow validates its question, retrieves evidence, calls the injected client, and
validates the answer against the retrieved IDs. Invalid model output is a service failure, not a
confident answer. TimeoutError and malformed output become a small application exception;
unrelated programming errors remain visible. The mock is deterministic, but real LLM output is
probabilistic even with a stable prompt, so keep quality evaluation separate from unit tests.


In [ ]:
class AIServiceError(RuntimeError):
    pass

def ask(question, client, k=2):
    if not isinstance(question,str) or not question.strip() or len(question) > 500:
        raise ValueError("Question must contain 1–500 characters")
    evidence = retrieve(question,k=k)
    try:
        raw = client.complete(question=question.strip(),evidence=evidence)
        if not isinstance(raw,str) or len(raw) > client.config.max_output_chars:
            raise ValueError("Invalid response size/type")
        return parse_answer(raw,{item["id"] for item in evidence})
    except (ValueError,TimeoutError) as error:
        raise AIServiceError("AI response unavailable or invalid") from error

answer = ask("leave approval",mock,k=1)
print(json.dumps(asdict(answer)))
assert answer.citations == ("leave:0",) and not answer.abstained
assert ask("astronaut galaxies",mock).abstained


## 8. Evaluate retrieval and failure paths separately

Evaluate retrieval before blaming generation: did the correct source reach the model?
Then measure citation validity, answer support, refusal/abstention behavior, latency, and cost.
These three fixtures are a smoke test, not a meaningful quality benchmark. Keep a separate
held-out question set for real evaluation and include paraphrases, no-evidence questions,
contradictory/dated documents, and malicious instructions embedded in retrieved text.


In [ ]:
fixtures = [("leave approval","leave"),("energy readings","energy"),("tender checklist","tender")]
correct = sum(retrieve(question,k=1)[0]["source"] == source for question,source in fixtures)
print({"retrieval_hit_at_1":correct/len(fixtures),"questions":len(fixtures)})
assert correct == 3

class BrokenClient(MockClient):
    def complete(self, **kwargs):
        return "not json"

class SlowClient(MockClient):
    def complete(self, **kwargs):
        raise TimeoutError("simulated wait limit")

for broken in [BrokenClient(),SlowClient()]:
    try:
        ask("leave approval",broken)
    except AIServiceError as error:
        print(str(error))
    else:
        raise AssertionError("Expected service failure")


## 9. Optional external client and API integration

The core above uses no network. To add a real provider, implement the same
`complete(question=..., evidence=...) -> str` method and retain `config`. The adapter should:

1. Read a key from an environment variable at runtime and fail clearly if absent; never print it.
2. Build separate instruction and evidence messages, naming the expected JSON schema and source IDs.
3. Set a timeout and output-token limit; extract text from the provider's response envelope.
4. Translate provider-specific transient failures into the application's failure policy with bounded retries.
5. Validate all output with `parse_answer`; provider structured-output options supplement this validation.

Provider request formats differ; this lesson deliberately makes no live call and invents no generic
provider endpoint. Select an official SDK and its current documentation when you choose a provider.
The existing mock and failure doubles let you test the adapter's application contract first.

For a FastAPI feature, move chunking/indexing into retrieval.py, the client boundary into clients.py,
schemas into schemas.py, and ask into service.py. A synchronous `/ask` handler can inject the client
with Depends, validate a bounded question with Pydantic, return an AnswerResponse, and translate
AIServiceError into HTTP 502 (upstream failure). Keep internal error details out of the response.
Do not rebuild the index for every request. Use access-filtered indexes in multi-user systems.
The commented optional cell below reuses stage 26 packages and still uses the mock.

## 10. Common mistakes and debugging

Check chunk text and retrieval scores before changing prompts. Empty/out-of-vocabulary queries
produce zero vectors in this toy system; they are not embedding API outages. Similarity is not
truth. A valid citation ID can accompany an unsupported sentence. Evaluate answer support manually
or with a reviewed rubric. JSON markdown fences are not JSON; require a raw JSON contract rather
than silently stripping arbitrary model text. Treat retrieved instructions as untrusted data,
restrict tools and permissions, and avoid putting secrets into model context.


In [ ]:
# Optional: requires requirements.txt. Uncomment as a block after installing it.
# from fastapi import FastAPI, Depends, HTTPException
# from fastapi.testclient import TestClient
# from pydantic import BaseModel, Field
# from typing import Annotated
# class AskRequest(BaseModel):
#     question: str = Field(min_length=1, max_length=500)
# class AnswerResponse(BaseModel):
#     answer: str
#     citations: list[str]
#     abstained: bool
# ai_app = FastAPI()
# def get_client():
#     return mock
# @ai_app.post("/ask", response_model=AnswerResponse)
# def ask_endpoint(body: AskRequest, client: Annotated[MockClient, Depends(get_client)]):
#     try:
#         return asdict(ask(body.question, client))
#     except ValueError as error:
#         raise HTTPException(status_code=422, detail="Invalid question") from error
#     except AIServiceError as error:
#         raise HTTPException(status_code=502, detail="AI service unavailable") from error
# with TestClient(ai_app) as client:
#     assert client.post("/ask", json={"question":"leave approval"}).json()["citations"] == ["leave:0"]


## Exercise 1

The stride is size minus overlap, so starts are 0,2,4; validation prevents a zero stride.


In [ ]:
assert chunk_words("one two three four five six",3,1) == ["one two three","three four five","five six"]
try:
    chunk_words("text",3,3)
except ValueError:
    pass
else:
    raise AssertionError("Invalid overlap accepted")


## Exercise 2

Normalization removes overall count magnitude; a word outside this vocabulary contributes nothing.


In [ ]:
practice_vocab = ["leave","energy"]
assert embed("leave leave",practice_vocab) == embed("leave",practice_vocab) == [1.,0.]
assert similarity(embed("leave",practice_vocab),embed("energy",practice_vocab)) == 0
assert similarity(embed("leave",practice_vocab),embed("leave leave",practice_vocab)) == 1


## Exercise 3

The retrieval contract distinguishes evidence candidates from no overlap, without claiming that overlap proves relevance.


In [ ]:
assert retrieve("tender checklist",1)[0]["source"] == "tender"
assert retrieve("astronaut galaxies") == []


## Exercise 4

JSON syntax and semantic type/citation checks fail at different points but both protect the application boundary.


In [ ]:
practice_bad = ["{",json.dumps({"answer":"x","citations":["unknown"],"abstained":False}),
                json.dumps({"answer":"x","citations":[],"abstained":"false"})]
for raw in practice_bad:
    try:
        parse_answer(raw,{"leave:0"})
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid output accepted")
assert parse_answer(json.dumps({"answer":"Unknown","citations":[],"abstained":True}),set()).abstained


## Exercise 5

A structurally correct response can still refer to evidence never retrieved. Input errors and upstream failures belong to different boundaries.


In [ ]:
class PracticeInventedClient(MockClient):
    def complete(self, **kwargs):
        return json.dumps({"answer":"Unsupported","citations":["invented:0"],"abstained":False})
try:
    ask("leave approval",PracticeInventedClient())
except AIServiceError as error:
    assert isinstance(error.__cause__,ValueError)
else:
    raise AssertionError("Invented citation accepted")
try:
    ask("   ",mock)
except ValueError:
    pass
else:
    raise AssertionError("Blank question accepted")


## Chapter project

Build `evaluate_assistant(client)` on four fixed cases: leave approval→leave, energy readings→energy,
tender checklist→tender, and astronaut galaxies→abstain. Return counts for passed, abstained, and
service_errors. Catch AIServiceError per case so one failed call does not discard other results.
Passing an answer requires a citation to the expected source (look up chunk IDs); unknown queries
must abstain. Do not count these fixture matches as proof of factual answer quality.

**Check:** MockClient gives passed=4, abstained=1, service_errors=0. BrokenClient gives
passed=0, abstained=0, service_errors=4. Write a short plan for a held-out evaluation set and a real
provider adapter, including timeouts, secret handling, answer support, and prompt-injection cases.

The implementation separates inputs, transformation, and reporting.


In [ ]:
def evaluate_assistant(client):
    cases = [("leave approval","leave"),("energy readings","energy"),
             ("tender checklist","tender"),("astronaut galaxies",None)]
    sources = {chunk["id"]:chunk["source"] for chunk in chunks}
    report = {"passed":0,"abstained":0,"service_errors":0}
    for question,expected in cases:
        try:
            result = ask(question,client,k=1)
        except AIServiceError:
            report["service_errors"] += 1
            continue
        report["abstained"] += int(result.abstained)
        passed = result.abstained if expected is None else (
            not result.abstained and any(sources[citation] == expected for citation in result.citations))
        report["passed"] += int(passed)
    return report
assert evaluate_assistant(MockClient()) == {"passed":4,"abstained":1,"service_errors":0}
assert evaluate_assistant(BrokenClient()) == {"passed":0,"abstained":0,"service_errors":4}
print(evaluate_assistant(MockClient()))
print("Next evaluation: unseen paraphrases, reviewed support labels, dated/conflicting policies, injection text; measure latency/cost separately.")
